<h2>Impact (v0.61) - 'Impact (v0.1).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 

'v0.61' makes use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# Define functions for plotting a count of values in a given data field.
int_arr_voys_field_count = lambda field: int_arr_voys[field].value_counts().to_frame().reset_index()
int_dep_voys_field_count = lambda field: int_dep_voys[field].value_counts().to_frame().reset_index()

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.61"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "516"
iso_country = "Namibia"
year = 2024
gdp = 13.460e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

'Output directory already exists.'

## 1.1 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv("/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_0.1.csv", dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_v0.61.csv".format(iso_country), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Namibia': 

Number of trade records present in dataset: 7607
Number of partner countries present in dataset: 131
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$3558.1m
Total weight within the dataset: 4082.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,516,Namibia,044,Bahamas,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,71.0,71.0,1159.641,1008.385,1159.641,Meat of bovine anima,0201 | Meat of bovine anima,Bahamas | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Bahamas | 02 | Meat and edible meat,201,6.6618,0.1501,both,0.061226,174.062114,[Container],11146.0,6925.801166,1205.519593
1,1,516,Namibia,392,Japan,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,14.4,14.4,162.514,125.815,162.514,Meat of bovine anima,0201 | Meat of bovine anima,Japan | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Japan | 02 | Meat and edible meat,201,6.6618,0.1501,both,0.088608,24.393351,[Container],16977.0,10549.015467,257.325841


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 603
Total size of the fleet: 321 vessels
8 vessel types contained in the dataset: ['Refrigerated bulk' 'General cargo' 'Container' 'Bulk carrier'
 'Chemical tanker' 'Vehicle' 'Oil tanker' 'Liquefied gas tanker']
Total transport work capacity: 79919.6 million deadweight-tonne nautical miles
Total cargo transported: 12986311.8 tonnes
Total transport work: 48153.3 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 8616.0 EJ
Total CO2 produced: 632850.7 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_arr_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_arr_voys_vess.shape[0] == 0:
                int_arr_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_arr_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
            
            if int_arr_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
                continue
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["lh_levy_usd_"+year] = lh_levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))


print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_im.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.lh_levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['9620 | Monopods, bipods, tr']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 7607
Number of partner countries present in dataset: 131

Total value of trade: US$3558.1m
Total weight within the dataset: 4082.1 million kilograms


Total FEM costs in 2041...

BAU: US$186.9m (5.3% ad-val)
Levy: US$374.1m (10.5% ad-val)
Levy (low-then-high): US$369.1m (10.4% ad-val)
'as-is': US$354.3m (10.0% ad-val)
Brazil: US$367.6m (10.3% ad-val)
LISCR: US$255.5m (7.2% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,516,Namibia,044,Bahamas,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,71.0,71.0,1159.641,1008.385,1159.641,Meat of bovine anima,0201 | Meat of bovine anima,Bahamas | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Bahamas | 02 | Meat and edible meat,201,6.6618,0.1501,both,0.061226,174.062114,[Container],11146.0,6925.801166,1205.519593,8.176061,8.141233,8.142379,8.143528,8.143528,8.167134,8.592394,16.387263,10.013319,9.607898,8.704667,8.740161,8.394425,15.472499,10.978445,9.906092,9.084387,8.889261,8.221028,17.049221,16.928400,12.448172,10.796163,9.993072,8.049737,18.144737,16.772698,14.422525,13.586039,10.568398,7.881557,18.406876,18.003800,17.107296,17.364755,11.554464,7.710205,18.115529,18.354254,17.581269,18.548930,13.552298,7.549091,16.177426,16.976981,16.451298,17.241153,13.123425,7.385433,14.443891,14.74306,14.871301,17.539200,14.306305
1,1,516,Namibia,392,Japan,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,14.4,14.4,162.514,125.815,162.514,Meat of bovine anima,0201 | Meat of bovine anima,Japan | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Japan | 02 | Meat and edible meat,201,6.6618,0.1501,both,0.088608,24.393351,[Container],16977.0,10549.015467,257.325841,1.745232,1.737798,1.738043,1.738288,1.738288,1.743327,1.834101,3.497966,2.137407,2.050867,1.858067,1.865643,1.791844,3.302703,2.343419,2.114518,1.939120,1.897469,1.754831,3.639265,3.613475,2.657142,2.304510,2.133085,1.718268,3.873110,3.580239,3.078580,2.900027,2.255892,1.682369,3.929065,3.843026,3.651661,3.706618,2.466374,1.645792,3.866875,3.917832,3.752834,3.959387,2.892824,1.611402,3.453175,3.623845,3.511634,3.680234,2.801279,1.576468,3.083141,3.14700,3.174374,3.743854,3.053772


## 2.4 Inpsect Output

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,198.8,195.5,192.6,189.7,186.9,184.0,181.3,178.6
1,Levy,321.7,311.0,343.5,376.4,374.1,362.2,326.6,304.6
2,Levy (LH),220.1,240.6,338.9,356.1,369.1,362.4,338.5,306.5
3,'as-is',211.6,223.1,259.5,303.9,354.3,353.2,334.7,310.7
4,Brazil,198.7,207.4,240.9,287.3,367.6,370.6,365.4,358.2
5,LISCR,199.3,204.3,227.3,237.2,255.5,296.0,291.3,300.5



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,5.6,5.5,5.4,5.3,5.3,5.2,5.1,5.0
1,Levy,9.0,8.7,9.7,10.6,10.5,10.2,9.2,8.6
2,Levy (LH),6.2,6.8,9.5,10.0,10.4,10.2,9.5,8.6
3,'as-is',5.9,6.3,7.3,8.5,10.0,9.9,9.4,8.7
4,Brazil,5.6,5.8,6.8,8.1,10.3,10.4,10.3,10.1
5,LISCR,5.6,5.7,6.4,6.7,7.2,8.3,8.2,8.4



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.477,1.453,1.431,1.410,1.389,1.367,1.347,1.327
1,Levy,2.390,2.310,2.552,2.796,2.779,2.691,2.426,2.263
2,Levy (LH),1.635,1.787,2.518,2.646,2.742,2.692,2.515,2.277
3,'as-is',1.572,1.658,1.928,2.257,2.632,2.624,2.486,2.309
4,Brazil,1.476,1.541,1.790,2.135,2.731,2.753,2.714,2.661
5,LISCR,1.481,1.518,1.689,1.762,1.898,2.199,2.164,2.233


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (%)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,122.9,115.5,150.9,186.6,187.2,178.1,145.3,126.0
1,Levy (LH),21.4,45.1,146.3,166.4,182.2,178.3,157.2,127.9
2,'as-is',12.9,27.6,66.9,114.1,167.4,169.2,153.3,132.1
3,Brazil,-0.1,11.9,48.3,97.6,180.7,186.5,184.0,179.6
4,LISCR,0.5,8.8,34.7,47.4,68.6,112.0,109.9,121.9



Total change in FEM costs by scenario (%)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,61.8,59.1,78.4,98.4,100.2,96.8,80.1,70.6
1,Levy (LH),10.8,23.0,76.0,87.7,97.5,96.9,86.7,71.6
2,'as-is',6.5,14.1,34.8,60.1,89.6,91.9,84.5,74.0
3,Brazil,-0.1,6.1,25.1,51.4,96.7,101.4,101.5,100.5
4,LISCR,0.3,4.5,18.0,25.0,36.7,60.8,60.6,68.3



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,3.5,3.2,4.2,5.2,5.3,5.0,4.1,3.5
1,Levy (LH),0.6,1.3,4.1,4.7,5.1,5.0,4.4,3.6
2,'as-is',0.4,0.8,1.9,3.2,4.7,4.8,4.3,3.7
3,Brazil,-0.0,0.3,1.4,2.7,5.1,5.2,5.2,5.0
4,LISCR,0.0,0.2,1.0,1.3,1.9,3.1,3.1,3.4



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.913,0.858,1.121,1.387,1.391,1.323,1.079,0.936
1,Levy (LH),0.159,0.335,1.087,1.236,1.353,1.325,1.168,0.950
2,'as-is',0.096,0.205,0.497,0.848,1.244,1.257,1.139,0.982
3,Brazil,-0.001,0.089,0.359,0.725,1.343,1.386,1.367,1.334
4,LISCR,0.004,0.065,0.258,0.352,0.510,0.832,0.817,0.906


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_v0.61.csv".format(iso_country), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Namibia': 

Number of trade records present in dataset: 3569
Number of partner countries present in dataset: 114
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2931.6m
Total weight within the dataset: 4310.7 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,516,Namibia,056,Belgium,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,72363.0,72363.0,772543.778,772543.778,772543.778,Meat of bovine anima,0201 | Meat of bovine anima,Belgium | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Belgium | 02 | Meat and edible meat,201,6.6618,0.1501,both,0.093668,115958.821078,[Container],10468.0,6504.511628,754255.500070
1,1,516,Namibia,180,Dem. Rep. of the Congo,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,55.0,55.0,217.547,217.547,217.547,Meat of bovine anima,0201 | Meat of bovine anima,Dem. Rep. of the Congo | 0201 | Meat of bovine...,02,02 | Meat and edible meat,Meat and edible meat,Dem. Rep. of the Congo | 02 | Meat and edible ...,201,6.6618,0.1501,both,0.252819,32.653805,[Container],2222.0,1380.686362,45.084663


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 603
Total size of the fleet: 320 vessels
8 vessel types contained in the dataset: ['Refrigerated bulk' 'General cargo' 'Container' 'Bulk carrier'
 'Chemical tanker' 'Vehicle' 'Oil tanker' 'Liquefied gas tanker']
Total transport work capacity: 59400.3 million deadweight-tonne nautical miles
Total cargo transported: 13009049.2 tonnes
Total transport work: 35763.5 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 5984.1 EJ
Total CO2 produced: 450567.1 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_dep_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_dep_voys_vess.shape[0] == 0:
                int_dep_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_dep_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

            if int_dep_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["lh_levy_usd_"+year] = lh_levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.lh_levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(
    trade_ex_na.primaryValue.sum(), 
    round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)
))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['9620 | Monopods, bipods, tr']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 3569
Number of partner countries present in dataset: 114

Total value of trade: US$2931.6m
Total weight within the dataset: 4310.7 million kilograms


Total FEM costs in 2041...

BAU: US$206.4m (7.0% ad-val)
Levy: US$414.1m (14.1% ad-val)
Levy (low-then-high): US$409.5m (14.0% ad-val)
'as-is': US$392.3m (13.4% ad-val)
Brazil: US$405.7m (13.8% ad-val)
LISCR: US$286.6m (9.8% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,HS4,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,516,Namibia,056,Belgium,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,72363.0,72363.0,772543.778,772543.778,772543.778,Meat of bovine anima,0201 | Meat of bovine anima,Belgium | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Belgium | 02 | Meat and edible meat,201,6.6618,0.1501,both,0.093668,115958.821078,[Container],10468.0,6504.511628,754255.500070,5104.487447,5082.657500,5083.374844,5084.093829,5084.093829,5098.904791,5364.565416,10235.964419,6252.505694,5999.265266,5434.245162,5456.452221,5240.891745,9662.320288,6854.85289,6184.507249,5671.517917,5549.773812,5132.569323,10647.681165,10575.470488,7772.252301,6740.365559,6239.020566,5025.558759,11332.029492,10473.991528,9005.313037,8483.486746,6597.583031,4920.490586,11495.496717,11244.937101,10681.791466,10842.912629,7213.796613,4813.441483,11314.005941,11463.964282,10981.344514,11583.056439,8461.589012,4712.793857,10103.843715,10604.473165,10273.179947,10766.257232,8196.107535,4610.555522,9019.497442,9205.645760,9287.197348,10949.995691,8930.776054
1,1,516,Namibia,180,Dem. Rep. of the Congo,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,55.0,55.0,217.547,217.547,217.547,Meat of bovine anima,0201 | Meat of bovine anima,Dem. Rep. of the Congo | 0201 | Meat of bovine...,02,02 | Meat and edible meat,Meat and edible meat,Dem. Rep. of the Congo | 02 | Meat and edible ...,201,6.6618,0.1501,both,0.252819,32.653805,[Container],2222.0,1380.686362,45.084663,0.305114,0.303809,0.303852,0.303895,0.303895,0.304781,0.320660,0.611842,0.373736,0.358598,0.324825,0.326152,0.313268,0.577553,0.40974,0.369671,0.339008,0.331731,0.306793,0.636452,0.632135,0.464576,0.402897,0.372930,0.300396,0.677358,0.626070,0.538281,0.507090,0.394362,0.294116,0.687129,0.672152,0.638490,0.648121,0.431196,0.287717,0.676280,0.685244,0.656396,0.692362,0.505781,0.281701,0.603944,0.633869,0.614066,0.643539,0.489912,0.275590,0.539129,0.550256,0.555130,0.654522,0.533826


## 3.4 Inspect Output

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,218.7,215.3,212.3,209.3,206.4,203.5,200.7,197.9
1,Levy,346.3,333.5,364.5,401.6,414.1,386.8,358.4,337.2
2,Levy (LH),240.5,258.4,376.5,396.9,409.5,394.6,374.0,339.5
3,'as-is',231.1,238.4,284.8,322.8,392.3,379.7,366.5,341.8
4,Brazil,218.1,227.3,262.3,307.3,405.7,399.9,396.0,390.1
5,LISCR,218.8,224.2,248.2,259.1,286.6,328.9,321.9,330.0



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,7.5,7.3,7.2,7.1,7.0,6.9,6.8,6.7
1,Levy,11.8,11.4,12.4,13.7,14.1,13.2,12.2,11.5
2,Levy (LH),8.2,8.8,12.8,13.5,14.0,13.5,12.8,11.6
3,'as-is',7.9,8.1,9.7,11.0,13.4,13.0,12.5,11.7
4,Brazil,7.4,7.8,8.9,10.5,13.8,13.6,13.5,13.3
5,LISCR,7.5,7.6,8.5,8.8,9.8,11.2,11.0,11.3



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.625,1.599,1.577,1.555,1.534,1.512,1.491,1.470
1,Levy,2.573,2.478,2.708,2.984,3.077,2.874,2.663,2.505
2,Levy (LH),1.787,1.920,2.797,2.949,3.043,2.932,2.778,2.522
3,'as-is',1.717,1.771,2.116,2.398,2.914,2.821,2.723,2.539
4,Brazil,1.621,1.689,1.949,2.283,3.014,2.971,2.942,2.898
5,LISCR,1.625,1.666,1.844,1.925,2.129,2.443,2.392,2.452


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (%)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,127.7,118.2,152.3,192.3,207.7,183.3,157.7,139.3
1,Levy (LH),21.8,43.2,164.2,187.6,203.1,191.1,173.3,141.6
2,'as-is',12.5,23.1,72.5,113.5,185.9,176.2,165.8,143.9
3,Brazil,-0.5,12.0,50.0,98.0,199.2,196.4,195.3,192.2
4,LISCR,0.1,9.0,35.9,49.8,80.2,125.4,121.2,132.1



Total change in FEM costs by scenario (%)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,58.4,54.9,71.7,91.9,100.6,90.1,78.6,70.4
1,Levy (LH),10.0,20.1,77.4,89.6,98.4,93.9,86.3,71.6
2,'as-is',5.7,10.7,34.2,54.2,90.0,86.6,82.6,72.7
3,Brazil,-0.2,5.6,23.6,46.8,96.5,96.5,97.3,97.1
4,LISCR,0.1,4.2,16.9,23.8,38.9,61.6,60.4,66.8



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,4.4,4.0,5.2,6.6,7.1,6.3,5.4,4.8
1,Levy (LH),0.7,1.5,5.6,6.4,6.9,6.5,5.9,4.8
2,'as-is',0.4,0.8,2.5,3.9,6.3,6.0,5.7,4.9
3,Brazil,-0.0,0.4,1.7,3.3,6.8,6.7,6.7,6.6
4,LISCR,0.0,0.3,1.2,1.7,2.7,4.3,4.1,4.5



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.949,0.878,1.131,1.429,1.543,1.362,1.172,1.035
1,Levy (LH),0.162,0.321,1.220,1.394,1.509,1.420,1.287,1.052
2,'as-is',0.093,0.172,0.539,0.843,1.381,1.309,1.232,1.069
3,Brazil,-0.004,0.090,0.372,0.728,1.480,1.459,1.451,1.428
4,LISCR,0.001,0.067,0.267,0.370,0.596,0.932,0.901,0.982
